# 07. Construcción de documentos de resumen diario

Este notebook construye la estructura documental que se utilizará posteriormente en MongoDB para almacenar un resumen por cada día del conjunto de datos.

Los datos minuto a minuto se recuperan desde PostgreSQL y se transforman en un documento diario que incluye información sobre cobertura temporal, irradiancia, meteorología, calidad de las mediciones, imputaciones y campos reservados para futuros resultados de modelos.

En esta fase no se insertan documentos en MongoDB. El objetivo es validar la estructura y los cálculos utilizando inicialmente una única fecha.

In [1]:
from src.mongodb.daily_summary import build_daily_document
from src.database.connection import get_database_engine
from src.database.load_daily_measurements import load_daily_measurements

engine = get_database_engine()

test_date = "2023-07-15"

df_day = load_daily_measurements(
    engine=engine,
    date=test_date,
)

print(f"Número de registros: {len(df_day):,}")
print(f"Fecha mínima: {df_day['fecha'].min()}")
print(f"Fecha máxima: {df_day['fecha'].max()}")
print(f"Número de columnas: {df_day.shape[1]}")
print(f"Fechas duplicadas: {df_day['fecha'].duplicated().sum()}")

Número de registros: 1,440
Fecha mínima: 2023-07-15 00:00:00
Fecha máxima: 2023-07-15 23:59:00
Número de columnas: 29
Fechas duplicadas: 0


In [15]:
daily_document = build_daily_document(
    df=df_day,
    dataset_version="v3",
)

print("Documento diario construido correctamente.")

Documento diario construido correctamente.


In [16]:
from pprint import pprint

pprint(
    daily_document,
    sort_dicts=False,
    width=120,
)

{'fecha': datetime.datetime(2023, 7, 15, 0, 0, tzinfo=datetime.timezone.utc),
 'dataset': {'version': 'v3', 'origen': 'PostgreSQL', 'tabla_origen': 'solar.measurements'},
 'periodo': {'ano': 2023, 'mes': 7, 'dia': 15, 'dia_semana': 5},
 'cobertura': {'numero_registros': 1440,
               'fecha_inicio': datetime.datetime(2023, 7, 15, 0, 0),
               'fecha_fin': datetime.datetime(2023, 7, 15, 23, 59),
               'fechas_duplicadas': 0,
               'periodo_solar': {'registros_dia': 867, 'registros_noche': 573}},
 'irradiancia': {'ghi': {'media': 6.622047222222222,
                         'mediana': 0.0,
                         'minimo': 0.0,
                         'maximo': 236.333,
                         'desviacion_estandar': 30.830683952388345,
                         'nulos': 0},
                 'dni': {'media': 4.247625694444444,
                         'mediana': 0.0,
                         'minimo': 0.0,
                         'maximo': 355.75,
     

In [17]:
assert (
    daily_document["cobertura"]["numero_registros"]
    == len(df_day)
)

periodo_summary = daily_document[
    "cobertura"
]["periodo_solar"]

assert (
    periodo_summary["registros_dia"]
    + periodo_summary["registros_noche"]
    == len(df_day)
)

assert (
    daily_document["cobertura"]["fechas_duplicadas"]
    == df_day["fecha"].duplicated().sum()
)

print("Cobertura diaria validada correctamente.")

Cobertura diaria validada correctamente.


In [18]:
for target in [
    "codigo_ghi",
    "codigo_dni",
    "codigo_dhi",
]:
    target_frequency = daily_document[
        "calidad"
    ][target]["frecuencia"]

    assert sum(target_frequency.values()) == len(df_day), (
        f"La distribución de {target} no suma "
        "el total de registros diarios."
    )

print("Distribuciones de calidad validadas correctamente.")

Distribuciones de calidad validadas correctamente.


In [19]:
processing = daily_document["procesamiento"]

for indicator_name in [
    "imputacion_meteorologica",
    "irradiancia_original_nula",
]:
    indicator = processing[indicator_name]

    indicator_total = (
        indicator["valor_0"]
        + indicator["valor_1"]
        + indicator["nulos"]
    )

    assert indicator_total == len(df_day), (
        f"El indicador {indicator_name} no suma "
        "el total de registros."
    )

print("Indicadores binarios validados correctamente.")

Indicadores binarios validados correctamente.


In [20]:
from bson import BSON

try:
    BSON.encode(daily_document)
    print("El documento es compatible con BSON.")
except Exception as exc:
    print("El documento contiene tipos incompatibles con MongoDB.")
    raise exc

El documento es compatible con BSON.


## Resultado

Se ha recuperado desde PostgreSQL un día completo de mediciones y se ha transformado en un documento de resumen diario compatible con MongoDB.

El documento incluye la cobertura temporal del día, estadísticas descriptivas de irradiancia y meteorología, variables físicas, distribuciones de los códigos de calidad e indicadores binarios sobre imputación meteorológica y presencia original de irradiancias nulas.

También se han preparado estructuras vacías para incorporar posteriormente las rutas de las gráficas, los resultados diarios de los modelos, las anomalías detectadas y las explicaciones automáticas. En esta fase todavía no se ha insertado ningún documento en MongoDB.